In [1]:
import os
import glob
import yaml

import pandas as pd
import tifffile
import zarr
import napari
import dask.array as da

from utils.utility_functions import single_channel_pyramid

In [2]:
# I/O

# read single-cell data
main = pd.read_csv(os.path.join(os.getcwd(), '../input/main.csv'))

# read OME-TIFF, segmentation outlines, and H&E channels
tif_path = os.path.join(os.getcwd(), '../input/CyCIF-1A_image.ome.tif')
seg_path = os.path.join(os.getcwd(), '../input/CyCIF-1A_seg_outlines.ome.tif')
he_path = os.path.join(os.getcwd(), '../input/CyCIF-1A_hema_eosin.ome.tif')

# import markers.csv
markers = pd.read_csv(os.path.join(os.getcwd(), '../input/CyCIF-1A_mcmicro_markers.csv'))

# import image contrast settings
contrast_dir = os.path.join(os.getcwd(), '../input/CyCIF-1A_cylinter_contrast_limits.yml')
contrast_limits = yaml.safe_load(open(contrast_dir))['setContrast']

# the parquet file at the path below is being read because "main.csv" 
# uses trimmed marker channel names as column headers that differ from the raw channel names used 
# in the markers.csv file, which is itself used to index channels in the OME-TIFF image.
for_channels = pd.read_parquet(
    os.path.join(os.getcwd(), '../input/CyCIF-1A_clean_cylinter_clustering_3d_leiden.parquet')
)

# isolate antibodies of interest
abx_channels = [i for i in for_channels.columns if 'nucleiRingMask' in i if 'Hoechst' not in i]

In [3]:
# add H&E image to Napari viewer
viewer = napari.Viewer()
img, _, _ = single_channel_pyramid(he_path, channel=0)
viewer.add_image(img, visible=False, name="H&E")

<Image layer 'H&E' at 0x1d75c3310>

In [4]:
# add DNA1 channel to image viewer
dna, min, max = single_channel_pyramid(glob.glob(tif_path)[0], channel=0)
viewer.add_image(
    dna, rgb=False, blending='additive',
    colormap='gray', visible=True, opacity=0.8,
    name='DNA1', contrast_limits=(min, max)
)

<Image layer 'DNA1' at 0x1d9de5150>

In [5]:
# add marker channels to image viewer and apply previously defined contrast limits
for ch in abx_channels:
    ch = ch.rsplit('_', 1)[0]
    channel_number = markers['channel_number'][markers['marker_name'] == ch]
    
    img, min, max = single_channel_pyramid(
        glob.glob(tif_path)[0], channel=(channel_number.item() - 1)
    )
    viewer.add_image(
        img, rgb=False, blending='additive', colormap='lime', visible=False, name=ch,
        contrast_limits=(min, max)
    )
for ch in abx_channels:
    ch = ch.rsplit('_', 1)[0]
    viewer.layers[ch].contrast_limits = (
        contrast_limits[ch][0], contrast_limits[ch][1])

In [6]:
centroids = main[['Y_centroid', 'X_centroid']][(main['Seg'] == 3) & (main['VAE9_VIG7'] == 2)]
viewer.add_points(
    centroids, name='Seg3_V2', face_color='#ff7f0e', border_color='white',
    border_width=0.0, size=60.0, opacity=1.0, blending='translucent', visible=False
)

centroids = main[['Y_centroid', 'X_centroid']][(main['Seg'] == 3) & (main['VAE9_VIG7'] == 7)]
viewer.add_points(
    centroids, name='Seg3_7', face_color='#ff9896', border_color='white',
    border_width=0.0, size=60.0, opacity=1.0, blending='translucent', visible=False
)

centroids = main[['Y_centroid', 'X_centroid']][(main['Seg'] == 3) & (main['VAE9_VIG7'] == 12)]
viewer.add_points(
    centroids, name='S3_V12', face_color='#e377c2', border_color='white',
    border_width=0.0, size=60.0, opacity=1.0, blending='translucent', visible=False
)

centroids = main[['Y_centroid', 'X_centroid']][(main['Seg'] == 3) & (main['VAE9_VIG7'] == 14)]
viewer.add_points(
    centroids, name='S3_V14', face_color='#55aaff', border_color='white',
    border_width=0.0, size=60.0, opacity=1.0, blending='translucent', visible=False
)

centroids = main[['Y_centroid', 'X_centroid']][(main['Seg'] == 3) & (main['VAE9_VIG7'] == 16)]
viewer.add_points(
    centroids, name='Seg3_V16', face_color='#bcbd22', border_color='white',
    border_width=0.0, size=60.0, opacity=1.0, blending='translucent', visible=False
)

<Points layer 'Seg3_V16' at 0x1e0b03d50>

In [7]:
# add segmentation outlines to image viewer
seg, min, max = single_channel_pyramid(glob.glob(seg_path)[0], channel=0)
viewer.add_image(
    seg, rgb=False, blending='additive',
    colormap='gray', visible=False,
    name='segmentation', opacity=0.3, contrast_limits=(min, max)
)

<Image layer 'segmentation' at 0x1e0f82fd0>

In [8]:
# run image viewer
viewer.scale_bar.visible = True
viewer.scale_bar.unit = 'um'